# 부록 08. Structured Output으로 정형 데이터 받기

에이전트가 JSON, Pydantic 모델 등 정형화된 형식으로 응답하도록 설정하는 방법을 학습합니다.

## 학습 목표
- Structured Output의 개념과 필요성 이해
- ToolStrategy와 ProviderStrategy 사용법
- Pydantic 모델을 활용한 출력 스키마 정의

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")

## 2. Structured Output이란?

### 왜 필요한가?
- LLM의 자연어 응답을 프로그래밍적으로 처리하기 어려움
- JSON 파싱 오류, 필드 누락 등의 문제 발생
- 일관된 데이터 구조가 필요한 경우 (API 응답, DB 저장 등)

### LangChain 1.0의 접근법
- `response_format` 파라미터로 출력 형식 지정
- Pydantic 모델, dataclass, TypedDict 지원

## 3. 기본 사용법: Pydantic 모델

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal
from langchain.agents import create_agent

# 출력 스키마 정의
class ContactInfo(BaseModel):
    """연락처 정보"""
    name: str = Field(description="사람 이름")
    email: str = Field(description="이메일 주소")
    phone: Optional[str] = Field(default=None, description="전화번호")

# 에이전트 생성
agent = create_agent(
    model="gpt-4o-mini",
    tools=[],
    system_prompt="텍스트에서 연락처 정보를 추출하세요.",
    response_format=ContactInfo  # Pydantic 모델 직접 전달
)

print("Structured Output 에이전트 생성 완료")

In [ ]:
# 테스트
result = agent.invoke({
    "messages": [{
        "role": "user", 
        "content": "김철수 씨 연락처: kim.cs@example.com, 010-1234-5678"
    }]
})

# structured_response 키에서 결과 추출
contact = result["structured_response"]
print(f"추출된 연락처:")
print(f"  이름: {contact.name}")
print(f"  이메일: {contact.email}")
print(f"  전화: {contact.phone}")
print(f"\n타입: {type(contact)}")

## 4. ToolStrategy: 도구 호출 기반

In [ ]:
from langchain.agents.structured_output import ToolStrategy

class TaskAnalysis(BaseModel):
    """작업 분석 결과"""
    task_name: str = Field(description="작업 이름")
    priority: Literal["high", "medium", "low"] = Field(description="우선순위")
    estimated_hours: float = Field(description="예상 소요 시간 (시간)")
    dependencies: list[str] = Field(default=[], description="선행 작업 목록")

# ToolStrategy 명시적 사용
agent_tool_strategy = create_agent(
    model="gpt-4o-mini",
    tools=[],
    system_prompt="작업을 분석하여 구조화된 정보를 추출하세요.",
    response_format=ToolStrategy(TaskAnalysis)
)

result = agent_tool_strategy.invoke({
    "messages": [{
        "role": "user",
        "content": "사용자 인증 시스템 구현: 로그인 기능 필요, 긴급, 약 8시간 소요 예상, 데이터베이스 설계가 먼저 완료되어야 함"
    }]
})

task = result["structured_response"]
print(f"작업: {task.task_name}")
print(f"우선순위: {task.priority}")
print(f"예상 시간: {task.estimated_hours}시간")
print(f"선행 작업: {task.dependencies}")

## 5. ProviderStrategy: 네이티브 구조화 출력

In [ ]:
from langchain.agents.structured_output import ProviderStrategy

class MovieReview(BaseModel):
    """영화 리뷰 분석"""
    title: str = Field(description="영화 제목")
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정 분류")
    rating: float = Field(ge=0, le=10, description="평점 (0-10)")
    summary: str = Field(description="리뷰 요약")

# ProviderStrategy: OpenAI의 JSON 모드 사용
agent_provider = create_agent(
    model="gpt-4o-mini",
    tools=[],
    system_prompt="영화 리뷰를 분석하세요.",
    response_format=ProviderStrategy(MovieReview)
)

result = agent_provider.invoke({
    "messages": [{
        "role": "user",
        "content": "인셉션은 정말 놀라운 영화였어요! 스토리가 복잡하지만 매력적이고, 시각 효과도 환상적이었습니다. 강력 추천합니다."
    }]
})

review = result["structured_response"]
print(f"영화: {review.title}")
print(f"감정: {review.sentiment}")
print(f"평점: {review.rating}/10")
print(f"요약: {review.summary}")

## 6. 복잡한 스키마: 중첩 모델

In [ ]:
from typing import List

class Address(BaseModel):
    """주소 정보"""
    street: str
    city: str
    country: str = "대한민국"

class Person(BaseModel):
    """개인 정보"""
    name: str
    age: int
    email: str
    address: Address  # 중첩 모델

class Company(BaseModel):
    """회사 정보"""
    name: str
    industry: str
    employees: List[Person]  # 리스트 of 중첩 모델
    headquarters: Address

# 복잡한 구조 추출
complex_agent = create_agent(
    model="gpt-4o-mini",
    tools=[],
    system_prompt="텍스트에서 회사 정보를 추출하세요.",
    response_format=Company
)

result = complex_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """테크스타트 (IT 산업)는 서울시 강남구 테헤란로에 본사가 있습니다.
        주요 직원으로는:
        - 김영희 (35세, yh.kim@techstart.com, 서울 서초구 거주)
        - 박철수 (28세, cs.park@techstart.com, 서울 송파구 거주)"""
    }]
})

company = result["structured_response"]
print(f"회사명: {company.name}")
print(f"산업: {company.industry}")
print(f"본사: {company.headquarters.city}")
print(f"\n직원 목록:")
for emp in company.employees:
    print(f"  - {emp.name} ({emp.age}세) - {emp.email}")

## 7. dataclass 사용

In [ ]:
from dataclasses import dataclass

@dataclass
class WeatherReport:
    """날씨 리포트"""
    city: str
    temperature: float
    condition: str
    humidity: int

# dataclass 사용 (dict 반환)
weather_agent = create_agent(
    model="gpt-4o-mini",
    tools=[],
    system_prompt="날씨 정보를 구조화하세요.",
    response_format=ToolStrategy(WeatherReport)
)

result = weather_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "서울은 현재 맑고, 기온 22도, 습도 45%입니다."
    }]
})

weather = result["structured_response"]
print(f"도시: {weather['city'] if isinstance(weather, dict) else weather.city}")
print(f"기온: {weather['temperature'] if isinstance(weather, dict) else weather.temperature}°C")

## 8. 도구와 함께 사용

In [ ]:
from langchain.tools import tool

@tool
def search_database(query: str) -> str:
    """데이터베이스에서 정보를 검색합니다."""
    # 시뮬레이션
    return f"'{query}'에 대한 검색 결과: 관련 데이터 3건 발견"

class SearchResult(BaseModel):
    """검색 결과 요약"""
    query: str = Field(description="검색 쿼리")
    found_count: int = Field(description="발견된 항목 수")
    summary: str = Field(description="결과 요약")

# 도구와 structured output 함께 사용
search_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_database],
    system_prompt="검색을 수행하고 결과를 구조화하세요.",
    response_format=ToolStrategy(SearchResult)
)

result = search_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "최근 AI 기술 동향을 검색해줘"
    }]
})

search_result = result["structured_response"]
print(f"검색어: {search_result.query}")
print(f"결과 수: {search_result.found_count}")
print(f"요약: {search_result.summary}")

## 9. 요약

### 이 노트북에서 배운 것

1. **Structured Output 기본**
   - `response_format` 파라미터로 출력 형식 지정
   - `structured_response` 키에서 결과 추출

2. **출력 전략**
   - `ToolStrategy`: 도구 호출 기반 (범용)
   - `ProviderStrategy`: 네이티브 JSON 모드 (지원 모델만)

3. **스키마 정의**
   - Pydantic `BaseModel` (추천)
   - `dataclass`
   - `TypedDict`

4. **고급 사용**
   - 중첩 모델
   - 도구와 함께 사용
   - 유효성 검증

### 다음 단계
- 부록_09: Streaming으로 실시간 응답 처리하기